# Capítulo 7 · Gestionar el contexto y el coste

Contar tokens y estimar el coste, medir cuánto tarda el modelo en leer y en escribir, comparar cinco formas de recordar una conversación larga y probar dos cachés: la de prefijos y la de respuestas.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. El cuaderno carga el modelo de lenguaje y el de embeddings. Tus tiempos serán distintos de los del libro, pero las proporciones deberían parecerse.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar

In [ ]:
"""Capítulo 7: gestionar el contexto y el coste. Modelo abierto, en GPU o CPU."""
import copy
import json
import re
import time
import unicodedata
import urllib.request
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-0.6B"             # sin GPU: más pequeño y más rápido
MODELO = None                              # None: elegir según el entorno
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar(nombre=None):
    """El modelo de lenguaje y el de embeddings, en la GPU si la hay."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if nombre is None:
        nombre = MODELO_GPU if dispositivo == "cuda" else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    modelo.to(dispositivo).eval()
    embeddings = SentenceTransformer(MODELO_EMBEDDINGS, device=dispositivo)
    print(f"Modelo {nombre} en {dispositivo}")
    return tok, modelo, embeddings


def descargar(archivo):
    """Un archivo de la carpeta datos; en Colab, se descarga del repositorio."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return ruta.read_text(encoding="utf-8")


def sincronizar():
    """Espera a que la GPU termine, para medir bien el tiempo."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [ ]:
tok, modelo, embeddings = cargar(MODELO)

## 2. Tokens, tiempo y dinero

In [ ]:
def tokens(texto):
    """Número de tokens de un texto."""
    return len(tok(texto, add_special_tokens=False)["input_ids"])


def tokens_conversacion(conversacion):
    """Tokens de la conversación completa, con la plantilla de chat."""
    texto = tok.apply_chat_template(conversacion, tokenize=False,
                                    add_generation_prompt=True,
                                    enable_thinking=False)
    return tokens(texto)


# Precios de ejemplo de una API comercial, en euros por millón de tokens.
PRECIO_ENTRADA, PRECIO_SALIDA = 0.40, 1.60


def coste(tokens_entrada, tokens_salida):
    """Euros que costaría una petición con los precios de ejemplo."""
    return (tokens_entrada * PRECIO_ENTRADA + tokens_salida * PRECIO_SALIDA) / 1e6

In [ ]:
for frase in ["¿Cuánto tiempo tengo para devolver una bicicleta si no me convence?",
              "How much time do I have to return a bike if I am not happy with it?"]:
    print(len(frase.split()), "palabras,", tokens(frase), "tokens:", frase)
condiciones = descargar("condiciones_generales.md")
print("Condiciones generales:", len(condiciones.split()), "palabras,", tokens(condiciones), "tokens")
print(f"Coste de leer 7384 tokens y escribir 416: {coste(7384, 416):.4f} euros")

In [ ]:
def generar(conversacion, max_tokens=150, cache=None):
    """Responde a una conversación. Devuelve la respuesta, los tokens de entrada
    y de salida y los segundos. Con cache, reutiliza el cálculo de un prefijo."""
    texto = tok.apply_chat_template(conversacion, tokenize=False,
                                    add_generation_prompt=True,
                                    enable_thinking=False)
    entrada = tok(texto, return_tensors="pt").input_ids.to(modelo.device)
    extra = {"past_key_values": copy.deepcopy(cache)} if cache is not None else {}
    sincronizar()
    inicio = time.time()
    with torch.no_grad():
        salida = modelo.generate(entrada, max_new_tokens=max_tokens, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tok.eos_token_id, **extra)
    sincronizar()
    nuevos = salida[0, entrada.shape[1]:]
    respuesta = tok.decode(nuevos, skip_special_tokens=True)
    respuesta = re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()
    return respuesta, entrada.shape[1], len(nuevos), time.time() - inicio


def tiempo_por_longitud(longitudes=(250, 500, 1000, 2000, 4000), salida=50):
    """Segundos para leer instrucciones de distinta longitud (con un solo token
    de salida) y para generar además 50 tokens. Cada medida se repite tres
    veces y se toma la mediana, para quitar ruido."""
    relleno = descargar("condiciones_generales.md") * 6
    filas = []
    for n in longitudes:
        ids = tok(relleno, add_special_tokens=False)["input_ids"][:n]
        texto = tok.decode(ids) + "\n\nResume el texto."
        mensajes = [{"role": "user", "content": texto}]
        generar(mensajes, max_tokens=1)                       # calentar
        leer = sorted(generar(mensajes, max_tokens=1)[3] for _ in range(3))[1]
        entrada = generar(mensajes, max_tokens=1)[1]
        medidas = sorted((generar(mensajes, max_tokens=salida)[2:] for _ in range(3)),
                         key=lambda m: m[1])
        generados, total = medidas[1]
        filas.append({"tokens": entrada, "leer": leer, "leer_y_generar": total,
                      "generados": generados})
        print(f"{entrada:5d} tokens: leer {leer:.2f} s; leer y generar "
              f"{generados} tokens {total:.2f} s")
    return filas

In [ ]:
tiempos = tiempo_por_longitud()

## 3. Una conversación larga y cinco memorias

In [ ]:
POLITICAS = (
    "Envíos: a la península en 48-72 horas, gratis en pedidos de más de 50 "
    "euros. A Canarias, en 5-7 días laborables, por 14,95 euros. "
    "Devoluciones: 30 días desde la entrega. Garantía: 2 años. Taller: revisión "
    "básica por 35 euros, la primera gratis para bicicletas compradas aquí. "
    "Pago a plazos sin intereses en compras de más de 300 euros. Horario de la "
    "tienda: de lunes a viernes de 10:00 a 20:00 y los sábados de 10:00 a 14:00.")


SISTEMA = ("Eres el asistente de atención al cliente de Ciclos Norte, una tienda "
           "de bicicletas. Responde en español, en una o dos frases. Puedes usar "
           "todo lo que el cliente te haya dicho en la conversación y estos datos "
           "de la tienda:\n" + POLITICAS)


# Cada turno del cliente y, si se comprueba algo, los textos que debe contener
# la respuesta (basta uno).
CONVERSACION = [
    ("Hola, soy Marta Iglesias. El mes pasado compré una bicicleta Aero 3, "
     "pedido CN-11020.", None),
    ("¿Cuánto tiempo tengo para devolverla si no me convence?", ["30 días"]),
    ("Vale. Otra cosa: ¿hacéis revisiones en el taller?", None),
    ("¿Y cuánto cuesta?", ["35 euros", "gratis", "gratuita"]),
    ("Por cierto, mi hermano vive en Tenerife. ¿Le podéis enviar un casco?", None),
    ("¿Cuánto tardaría en llegarle?", ["5-7", "5 a 7"]),
    ("Perdona, he perdido el correo de confirmación. ¿Cuál era el número de "
     "mi pedido?", ["CN-11020"]),
    ("¿Me recuerdas qué bici te dije que tenía? Quiero comprarle accesorios.",
     ["Aero 3"]),
    ("¿Puedo pagar los accesorios a plazos?", ["300"]),
    ("¿Abrís los sábados?", ["10:00", "14:00", "10 de la mañana"]),
    ("Una última cosa: ¿a nombre de quién está el pedido?", ["Marta"]),
    ("Gracias, eso es todo.", None),
]


RESUMIR = ("Resume en tres o cuatro frases los datos importantes de esta "
           "conversación de atención al cliente: el nombre del cliente, sus "
           "pedidos y productos, y lo que ha preguntado. Copia exactamente los "
           "nombres y los números.")


EXTRAER = ("Extrae de los mensajes del cliente estos datos y responde solo con un "
           'objeto JSON: {"nombre": ..., "pedido": ..., "producto": ...}. '
           "Usa null para los que no aparezcan.")

In [ ]:
def memoria_vacia(historial, estado):
    """Nada: solo el mensaje actual."""
    return []


def memoria_completa(historial, estado):
    """Todos los mensajes anteriores."""
    return historial


def memoria_ventana(historial, estado, turnos=2):
    """Solo los últimos turnos (cada turno son dos mensajes)."""
    return historial[-2 * turnos:]


def memoria_resumen(historial, estado, turnos=2):
    """Los últimos turnos y un resumen de los anteriores, que se actualiza cada
    vez que un turno sale de la ventana."""
    antiguos = historial[:-2 * turnos]
    if len(antiguos) > estado.get("resumidos", 0):
        texto = "\n".join(f"{m['role']}: {m['content']}" for m in antiguos)
        previo = estado.get("resumen", "")
        estado["resumen"] = generar(
            [{"role": "system", "content": RESUMIR},
             {"role": "user", "content": f"{previo}\n{texto}".strip()}],
            max_tokens=120)[0]
        estado["resumidos"] = len(antiguos)
        estado["llamadas_extra"] = estado.get("llamadas_extra", 0) + 1
    if not antiguos:
        return historial
    return ([{"role": "system", "content": "Resumen de la conversación anterior: "
              + estado["resumen"]}] + historial[-2 * turnos:])


def memoria_datos(historial, estado, turnos=2):
    """Los últimos turnos y unos pocos datos clave del cliente, extraídos por el
    modelo en JSON y acumulados. Un dato solo se guarda si aparece tal cual en
    el mensaje, si el pedido tiene el formato CN-12345 y si no se había guardado
    antes: así no entran valores inventados ni se pisan los buenos."""
    datos = estado.setdefault("datos", {})
    if historial:
        ultimo = historial[-2]["content"]          # último mensaje del cliente
        try:
            nuevo = json.loads(re.search(r"\{.*\}", generar(
                [{"role": "system", "content": EXTRAER},
                 {"role": "user", "content": ultimo}], max_tokens=60)[0], re.S)[0])
            for clave, valor in nuevo.items():
                valido = (isinstance(valor, str) and valor.lower() in ultimo.lower()
                          and (clave != "pedido" or re.fullmatch(r"CN-\d{5}", valor)))
                if valido and clave not in datos:
                    datos[clave] = valor
        except (TypeError, ValueError):
            pass
        estado["llamadas_extra"] = estado.get("llamadas_extra", 0) + 1
    ficha = json.dumps(datos, ensure_ascii=False)
    return ([{"role": "system", "content": "Datos del cliente: " + ficha}]
            + historial[-2 * turnos:])


def conversar(memoria):
    """Recorre la conversación con una memoria y comprueba las respuestas."""
    historial, estado, filas = [], {}, []
    for mensaje, claves in CONVERSACION:
        contexto = memoria(historial, estado)
        conversacion = ([{"role": "system", "content": SISTEMA}] + contexto
                        + [{"role": "user", "content": mensaje}])
        respuesta, entrada, salida, segundos = generar(conversacion)
        historial += [{"role": "user", "content": mensaje},
                      {"role": "assistant", "content": respuesta}]
        bien = None if claves is None else any(c.lower() in respuesta.lower()
                                               for c in claves)
        filas.append({"cliente": mensaje, "respuesta": respuesta, "bien": bien,
                      "tokens_entrada": entrada, "segundos": segundos})
    return filas, estado


MEMORIAS = {"Sin memoria": memoria_vacia,
            "Historial completo": memoria_completa,
            "Ventana de 2 turnos": memoria_ventana,
            "Resumen + ventana": memoria_resumen,
            "Datos clave + ventana": memoria_datos}

In [ ]:
memorias = {}
for nombre, memoria in MEMORIAS.items():
    filas, estado = conversar(memoria)
    memorias[nombre] = (filas, estado)
    bien = sum(f["bien"] for f in filas if f["bien"] is not None)
    entrada = sum(f["tokens_entrada"] for f in filas)
    print(f"{nombre:22s} {bien}/8 bien, {entrada} tokens de entrada, "
          f"{estado.get('llamadas_extra', 0)} llamadas extra")

Lee la conversación completa con una de las memorias, con una marca en las respuestas que fallan:

In [ ]:
filas, estado = memorias["Datos clave + ventana"]
for f in filas:
    marca = "✗ " if f["bien"] is False else "  "
    print(f"{marca}Cliente: {f['cliente']}\n   Asistente: {f['respuesta']}")
print("Estado final de la memoria:", estado)

## 4. Dos cachés

In [ ]:
def con_cache_de_prefijo(sistema, preguntas, max_tokens=40):
    """Responde a varias preguntas con el mismo mensaje de sistema, con y sin
    reutilizar el cálculo de ese prefijo."""
    prefijo = tok.apply_chat_template([{"role": "system", "content": sistema}],
                                      tokenize=False)
    ids = tok(prefijo, return_tensors="pt").input_ids.to(modelo.device)
    with torch.no_grad():
        cache = modelo(ids, use_cache=True).past_key_values
    resultados = {}
    for nombre, c in (("sin caché", None), ("con caché", cache)):
        generar([{"role": "user", "content": "Hola"}], max_tokens=1)   # calentar
        respuestas, total = [], 0
        for p in preguntas:
            r, _, _, s = generar([{"role": "system", "content": sistema},
                                  {"role": "user", "content": p}], max_tokens, c)
            respuestas.append(r)
            total += s
        resultados[nombre] = {"segundos": total, "respuestas": respuestas}
        print(f"{nombre}: {total:.2f} s para {len(preguntas)} preguntas")
    return ids.shape[1], resultados

In [ ]:
catalogo = json.loads(descargar("catalogo.json"))
sistema = ("Eres el asistente de Ciclos Norte. Responde en una frase con los "
           "datos de este catálogo:\n" + "\n".join(
               f"{p['nombre']} ({p['ref']}), {p['precio']} euros. {p['descripcion']}"
               for p in catalogo))
preguntas = ["¿Cuánto cuesta la Aero 3?", "¿Qué autonomía tiene la E-Plegable?",
             "¿Tenéis cascos para niños?", "¿Cuánto pesa la Plegable 20?",
             "¿Qué candado es más seguro?", "¿Para qué edad es la Junior 24?"]
largo, prefijo = con_cache_de_prefijo(sistema, preguntas)
print("Tokens del prefijo:", largo)
print("¿Respuestas iguales?", prefijo["sin caché"]["respuestas"] == prefijo["con caché"]["respuestas"])

In [ ]:
def normalizar(texto):
    """Minúsculas, sin tildes ni signos, para comparar preguntas."""
    texto = "".join(c for c in unicodedata.normalize("NFD", texto.lower())
                    if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9 ]", "", texto).strip()


# Preguntas que llegan una tras otra, con la intención a la que corresponden
# (para saber si una respuesta guardada sirve o no).
LLEGADAS = [
    ("¿Cuánto tarda un envío a Canarias?", "canarias"),
    ("¿Cuánto cuesta la revisión del taller?", "revision"),
    ("¿Cuánto tarda un envío a Canarias?", "canarias"),
    ("¿Puedo pagar a plazos?", "plazos"),
    ("¿cuánto tarda un envío a canarias", "canarias"),
    ("¿Cuántos días tengo para devolver un producto?", "devolucion"),
    ("¿Se puede pagar a plazos?", "plazos"),
    ("¿Cuánto cuesta el envío a Canarias?", "canarias_precio"),
    ("¿Qué precio tiene revisar la bici en vuestro taller?", "revision"),
    ("¿Abrís los sábados?", "sabados"),
    ("¿Abrís los domingos?", "domingos"),
    ("¿Hasta cuándo puedo devolver lo que compré?", "devolucion"),
    ("¿Puedo pagar a plazos?", "plazos"),
    ("¿Cuánto tarda un envío a Baleares?", "baleares"),
    ("¿Los sábados está abierta la tienda?", "sabados"),
    ("¿Cuánto cuesta la revisión del taller?", "revision"),
]


def probar_caches(umbral=0.93):
    """Caché exacta (mismo texto normalizado) frente a caché semántica
    (embedding de la pregunta más parecida por encima de un umbral)."""
    resultados = {}
    for tipo in ("exacta", "semántica"):
        guardadas, aciertos, errores = [], 0, 0
        for pregunta, intencion in LLEGADAS:
            v = embeddings.encode(["query: " + pregunta], normalize_embeddings=True)[0]
            encontrada = None
            if tipo == "exacta":
                encontrada = next((i for p, i, _ in guardadas
                                   if p == normalizar(pregunta)), None)
            elif guardadas:
                sims = [float(v @ w) for _, _, w in guardadas]
                if max(sims) >= umbral:
                    encontrada = guardadas[sims.index(max(sims))][1]
            if encontrada is None:             # no está: se llama al modelo
                guardadas.append((normalizar(pregunta), intencion, v))
            elif encontrada == intencion:
                aciertos += 1
            else:
                errores += 1
        resultados[tipo] = {"aciertos": aciertos, "errores": errores,
                            "llamadas_al_modelo": len(guardadas)}
        print(f"Caché {tipo}: {aciertos} respuestas reutilizadas bien, "
              f"{errores} mal, {len(guardadas)} llamadas al modelo de {len(LLEGADAS)}")
    return resultados

In [ ]:
caches = probar_caches()

## Ejercicios

1. Explica con tus palabras por qué un modelo de lenguaje no recuerda las conversaciones y cómo lo resuelven las aplicaciones.

2. Cuenta los tokens de diez frases en español y de su traducción al inglés. ¿Cuántos tokens por palabra salen en cada idioma? ¿Qué consecuencia tiene para el coste de una aplicación en español?

3. Con los precios de ejemplo, calcula el coste mensual del asistente con cada memoria, suponiendo 10 000 conversaciones de doce mensajes al mes. No olvides las llamadas extra del resumen y de la ficha.

4. Repite la medida del tiempo en CPU, en Colab sin GPU. ¿Cómo cambian las proporciones entre leer y generar?

5. Cambia la ventana deslizante a 4 y a 6 turnos. ¿Cuántas comprobaciones supera? ¿Cuántos tokens gasta? ¿Desde qué tamaño se parece al historial completo?

6. Añade a la conversación de Marta, después del mensaje 8, un mensaje que cambie un dato: «Perdona, me equivoqué: la bici es una Aero 5». ¿Qué memorias lo recogen? ¿Qué pasa con la ficha, que no sustituye los datos guardados? Propón una regla mejor.

7. Mide el ahorro de la caché de prefijos con respuestas de 10, 40 y 150 tokens. ¿Cuándo compensa más?

8. Prueba la caché semántica con umbrales de 0,90, 0,95 y 0,98. ¿Hay alguno que reutilice las paráfrasis sin confundir los sábados con los domingos?

9. Escribe un enrutador sencillo para el clasificador del capítulo 2: usa Qwen3-0.6B con diez ejemplos y, si su respuesta no es una categoría válida, repite la petición con Qwen2.5-1.5B. ¿Cuántas peticiones llegan al modelo grande? ¿Cuánto tiempo se ahorra?

10. **Presupuesto del asistente.** Escribe un breve informe con el coste estimado de atender 10 000 conversaciones al mes con el asistente de los capítulos 5 y 6 y la memoria que elijas, con los precios de ejemplo, y propón tres medidas para reducirlo.

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) y [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base). *Lost in the middle*: [Liu y colaboradores (2023)](https://arxiv.org/abs/2307.03172). Caché KV y vLLM: [Kwon y colaboradores (2023)](https://arxiv.org/abs/2309.06180). Lecciones 11.05, 11.11 y 11.15 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte y la conversación son ficticias. Los precios son de ejemplo. Código y experimentos propios.